### Bonus Solution: Exploring the Softmax Function

In [1]:
# Seting up the environment and importing necessary libraries

import torch

For Task 1, we should convert the softmax function (when $N = 2$) to the sigmoid function:


$$\sigma(z) = \frac{1}{1 + e^{-z}}, \quad
\text{softmax}(z) =
\begin{bmatrix}
\frac{e^{z_1}}{e^{z_1} + e^{z_2}} \\
\frac{e^{z_2}}{e^{z_1} + e^{z_2}}
\end{bmatrix}$$



Note that the two sides are fairly similar. It suffices to divide both the numerator and denominator for the
softmax function by $e^{z_1}$ (or $e^{z_2}$ ):

$$\text{softmax}(z) =
\begin{bmatrix}
\frac{e^{z_1}}{e^{z_1} + e^{z_2}} \\
\frac{e^{z_2}}{e^{z_1} + e^{z_2}}
\end{bmatrix} = 
\begin{bmatrix}
\frac{1}{1 + e^{z_2 - z_1}} \\
\frac{1}{1 + e^{z_1 - z_2}}
\end{bmatrix} = 
\begin{bmatrix}
\sigma(z_2 - z_1) \\
\sigma(z_1 - z_2)
\end{bmatrix}$$

To implement `mysoftmax`, we would have to call `torch.sigmoid` for the tensors $z_2−z_1$ and $z_1−z_2$,
and concatenate them together (this can be done using `torch.hstack`). Alternatively, we could call
`torch.sigmoid` only once, as the probabilities sum to $1$.

In [2]:
def mysoftmax(z):
    softmax_class0 = torch.sigmoid(z[:, 0:1] - z[:, 1:2])
    return torch.hstack([softmax_class0, 1 - softmax_class0])

In [3]:
# Sample test case.

torch.manual_seed(2109)
z = torch.randn(5, 2)

z_correct = z.clone().detach()
z_correct = torch.softmax(z_correct, dim=1)
print("Softmax of z:\n", z_correct)

z_test = z.clone().detach()
z_test = mysoftmax(z_test)
print("Softmax of z (implemented using sigmoid):\n", z_test)

assert z_test.shape == z_correct.shape, "Output shape does not match"
assert torch.all(torch.isclose(z_test, z_correct), dim=(0,1)), \
    "Output does not match"

Softmax of z:
 tensor([[0.6214, 0.3786],
        [0.2190, 0.7810],
        [0.1938, 0.8062],
        [0.7362, 0.2638],
        [0.1739, 0.8261]])
Softmax of z (implemented using sigmoid):
 tensor([[0.6214, 0.3786],
        [0.2190, 0.7810],
        [0.1938, 0.8062],
        [0.7362, 0.2638],
        [0.1739, 0.8261]])


In [4]:
# Large test case. 

torch.manual_seed(3264)
z_eval = torch.randn(100, 2)

z_eval_correct = z_eval.clone().detach()
z_eval_correct = torch.softmax(z_eval_correct, dim=1)

z_eval_test = z_eval.clone().detach()
z_eval_test = mysoftmax(z_eval_test)

assert torch.all(torch.isclose(z_eval_test, z_eval_correct), dim=(0,1)), \
    "Output does not match"

print("Large test case passed. Congratulations!")

Large test case passed. Congratulations!


For Task 2, we can differentiate softmax just like how we differentiate the sigmoid function.
- If $i=k$, by quotient rule,
$$\text{softmax}'(z)_{ik} =
\frac{e^{z_i}\left(\sum_{j=1}^{k} e^{z_j}\right) - e^{z_i}e^{z_k}}
{\left(\sum_{j=1}^{k} e^{z_j}\right)^2}$$
- If $i \not= k$, by quotient rule,
$$\text{softmax}'(z)_{ik} =
\frac{- e^{z_i}e^{z_k}}
{\left(\sum_{j=1}^{k} e^{z_j}\right)^2}$$

We can define $\delta_{ik}$ as $1$ if $i = k$, or $0$ if $i \not= k$. Then,
$$\text{softmax}'(z)_{ik} =
\frac{\delta_{ik}e^{z_i}\left(\sum_{j=1}^{k} e^{z_j}\right) - e^{z_i}e^{z_k}}
{\left(\sum_{j=1}^{k} e^{z_j}\right)^2} = 
\frac{e^{z_i}}
{\sum_{j=1}^{k} e^{z_j}}  \cdot
\left(\delta_{ik} - \frac{e^{z_k}}
{\sum_{j=1}^{k} e^{z_j}}\right)
$$

Notice that this is exactly $softmax(z)_i$ and $softmax(z)_k$! We can summarize our result as
$$\text{softmax}'(z)_{ik} = \text{softmax}(z)_{i} \cdot (\delta_{ik}-\text{softmax}(z)_{k})$$
and this can be easily implemented using broadcasting:

In [5]:
def mysoftmax_grad(z):
    n, m = z.shape
    z = torch.softmax(z, dim=1)
    return z.reshape(n, m, 1) * (torch.eye(m).reshape(1, m, m) - z.reshape(n, 1, m))

In [6]:
# Sample test case.

torch.manual_seed(2109)
z = torch.randn(2, 3)

softmax_fn = lambda z: torch.softmax(z, dim=1)

z_correct = z.clone().detach().requires_grad_(True)
z_correct = torch.autograd.functional.jacobian(softmax_fn, z_correct)
z_correct = z_correct.diagonal(dim1=0, dim2=2).permute((2, 0, 1))
print("Softmax derivative of z:\n", z_correct)

z_test = z.clone().detach()
z_test = mysoftmax_grad(z_test)
print("Softmax derivative of z (from the formula):\n", z_test)

assert z_test.shape == z_correct.shape, "Output shape does not match"
assert torch.all(torch.isclose(z_test, z_correct), dim=(0,1,2)), \
    "Output does not match"

Softmax derivative of z:
 tensor([[[ 0.2420, -0.2115, -0.0305],
         [-0.2115,  0.2301, -0.0186],
         [-0.0305, -0.0186,  0.0491]],

        [[ 0.1892, -0.0367, -0.1525],
         [-0.0367,  0.1238, -0.0871],
         [-0.1525, -0.0871,  0.2396]]])
Softmax derivative of z (from the formula):
 tensor([[[ 0.2420, -0.2115, -0.0305],
         [-0.2115,  0.2301, -0.0186],
         [-0.0305, -0.0186,  0.0491]],

        [[ 0.1892, -0.0367, -0.1525],
         [-0.0367,  0.1238, -0.0871],
         [-0.1525, -0.0871,  0.2396]]])


In [ ]:
# Large test case. 

torch.manual_seed(3264)
z = torch.randn(50, 10)

softmax_fn = lambda z: torch.softmax(z, dim=1)

z_correct = z.clone().detach().requires_grad_(True)
z_correct = torch.autograd.functional.jacobian(softmax_fn, z_correct)
z_correct = z_correct.diagonal(dim1=0, dim2=2).permute((2, 0, 1))

z_test = z.clone().detach()
z_test = mysoftmax_grad(z_test)

assert z_test.shape == z_correct.shape, "Output shape does not match"
assert torch.all(torch.isclose(z_test, z_correct), dim=(0,1,2)), \
    "Output does not match"

print("Large test case passed. Congratulations!")

Large test case passed. Congratulations!


$\delta$ would be the identity matrix, which can be obtained using $torch.eye$. The broadcasting mechanism is
similar to Lab 1. 

For Task 3, observe that the softmax function ensures that all output probabilities sums up to $1$. It is a
good idea to use the softmax function if the classes are mutually exclusive. On the other hand, use the
sigmoid function if the classes are independent events.